# SoccerNet player detection (YOLO export + train)

Exports MOT `gt.txt` + `gameinfo.ini` to Ultralytics YOLO format (class **player** only), then trains with **`yolo detect train`**.

On **your laptop**, run **`scripts/setup_soccernet_yolo_local.py`** from the repo (after `pip install -e .` and `pip install SoccerNet`) to **download SoccerNet tracking, unzip the official archives, and export YOLO** in one shot. On **Colab**, paths match [`colab_atharv.ipynb`](colab_atharv.ipynb); use that notebook’s downloader if `tracking/` is empty.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Same layout as colab_atharv.ipynb
DATA_DIR = '/content/drive/MyDrive/soccernet'
TRACKING_ROOT = f'{DATA_DIR}/tracking'
REPO_DIR = '/content/drive/MyDrive/repo'

# YOLO dataset + Ultralytics runs (on Drive so they persist)
YOLO_DATASET = f'{DATA_DIR}/yolo_player_export'

!mkdir -p {DATA_DIR}

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/atharvjain05/cv-project.git {REPO_DIR}
%cd {REPO_DIR}
!git pull

## SoccerNet tracking data

- **Colab / Drive:** If `tracking/train` is empty, use the **`SoccerNetDownloader`** cells in [`colab_atharv.ipynb`](colab_atharv.ipynb) (`!pip install SoccerNet`, `downloadDataTask` for `tracking`, then **unzip** any `*.zip` under `tracking/` if sequences still don’t appear).
- **Local:** From the repo root, run **`python scripts/setup_soccernet_yolo_local.py --help`** — it downloads the official zips, **extracts** them to `tracking/<split>/SNMOT-*`, and runs the YOLO exporter (needs `pip install -e .`, `SoccerNet`, and your NDA **password** / `SOCCERNET_PASSWORD`).

In [ ]:
# Colab-only fallback: same download snippet lives in colab_atharv.ipynb (SoccerNetDownloader + downloadDataTask).
# Prefer scripts/setup_soccernet_yolo_local.py on your machine for download + unzip + YOLO export.

## Install project + Ultralytics

`soccernet_detect` lives in the repo; detection needs **`ultralytics`** (see `requirements-detect.txt`).

In [ ]:
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
!pip install -q -r requirements-detect.txt
!pip install -q -e .

import sys
src = f'{REPO_DIR}/src'
if src not in sys.path:
    sys.path.insert(0, src)

## Check tracking split

Expect `train/SNMOT-*/img1/*.jpg` and `gt/gt.txt`.

In [ ]:
import os
from pathlib import Path

for split in ['train', 'test', 'challenge']:
    d = Path(TRACKING_ROOT) / split
    if d.is_dir():
        seqs = sorted(p.name for p in d.iterdir() if p.is_dir())
        print(f'{split}: {len(seqs)} sequences — {seqs[:3]}{"..." if len(seqs) > 3 else ""}')
    else:
        print(f'{split}: not found under {TRACKING_ROOT}')

## No-export player+ball detector

This trains a small heatmap detector directly from SoccerNet's native `img1/` frames and MOT `gt.txt`. It predicts both **players** and **ball** without YOLO labels, copied images, or a mirrored dataset folder.

Start with `--frame-stride 10 --max-frames 2000` as a smoke run; lower the stride or remove `--max-frames` for a fuller training run.

In [ ]:
%cd {REPO_DIR}
!python -m soccernet_detect.train_heatmap_tracker \
  --tracking-root {TRACKING_ROOT} \
  --split train \
  --out /content/heatmap_tracker \
  --epochs 5 \
  --imgsz 512 \
  --batch 8 \
  --base-channels 42 \
  --frame-stride 10 \
  --max-frames 2000 \
  --device cuda

## Faster path: build the YOLO dataset locally, upload once

Exporting with **`--out` on Drive** is slow because every JPEG copy goes through the Drive mount. The usual fix:

0. **All-in-one (download + YOLO export):** from repo root, `pip install SoccerNet`, set `SOCCERNET_PASSWORD`, then `python scripts/setup_soccernet_yolo_local.py` (see `--help`).
1. Or on your **Mac / Linux PC** (or WSL): same repo + `pip install -r requirements-detect.txt` + `pip install -e .`, with SoccerNet **`tracking/`** on a **local disk**.
2. Run the exporter with **`--out`** on **local SSD**, e.g. `~/data/yolo_player_export`, and **`--copy-images`** (or symlinks if you prefer on Unix).
3. **Archive** the folder: `cd ~/data && zip -r yolo_player_export.zip yolo_player_export` (or `tar -czvf …`).
4. **Upload** the zip to Drive (browser, Drive desktop app, or `rclone`).
5. In Colab: unzip to **`/content`** for fast training reads, or unzip under Drive if you only need storage:

```python
# Example after uploading yolo_player_export.zip to Drive
!unzip -q /content/drive/MyDrive/soccernet/yolo_player_export.zip -d /content
YOLO_DATASET = '/content/yolo_player_export'  # then point yolo train data=.../data.yaml here
```

The **`data.yaml`** inside the export uses absolute `path:` to wherever you built it. After unzip, either **edit `path:`** in `data.yaml` to the new root, or unzip so the folder name matches what you had locally (simplest: keep the same top-level folder name `yolo_player_export` and set `path` inside yaml — see note below).

If training fails with “image not found”, open **`data.yaml`** and set **`path:`** to the directory that **contains** `images/` and `labels/` (e.g. `/content/yolo_player_export`).

## Export MOT → YOLO dataset

- **Player-only** boxes: same `gameinfo.ini` filtering as trajectory training (ball / referee / staff dropped).
- **`--copy-images`** works better on Google Drive than symlinks.
- **Re-runs skip** re-copying when the destination JPEG **path already exists** (labels are still rewritten). Use **`--force-recopy`** to replace every image.
- Tune **`--frame-stride`** (e.g. `5`) for quick tests; use `1` for full data.

Writes **`data.yaml`** under `YOLO_DATASET`.

In [ ]:
%cd {REPO_DIR}
!python -m soccernet_detect.export_yolo \
  --tracking-root {TRACKING_ROOT} \
  --split train \
  --out {YOLO_DATASET} \
  --val-fraction 0.15 \
  --frame-stride 1 \
  --copy-images


## Train YOLO (Ultralytics)

Uses **`data.yaml`** from the export step. On an H200 you can raise **`batch`** or use a larger model (`yolo11s.pt`). Lower **`imgsz`** (e.g. 960) if CUDA OOM.

In [ ]:
%cd {REPO_DIR}
!yolo detect train model=yolo11n.pt data={YOLO_DATASET}/data.yaml epochs=50 imgsz=1280 batch=16 device=0

### Optional: train via Python API

Same args as `yolo detect train`; checkpoints land under `runs/detect/train*` by default.

In [ ]:
# from ultralytics import YOLO
# model = YOLO('yolo11n.pt')
# model.train(data=f'{YOLO_DATASET}/data.yaml', epochs=50, imgsz=1280, batch=16, device=0)

## Sanity check (label count)

Non-empty label files should roughly match images; many empties can mean missing `gt`/`gameinfo` paths.

In [ ]:
from pathlib import Path

for split in ['train', 'val']:
    lbl = Path(YOLO_DATASET) / 'labels' / split
    if not lbl.is_dir():
        print(split, 'no labels dir')
        continue
    files = list(lbl.glob('*.txt'))
    nonempty = sum(1 for f in files if f.read_text().strip())
    print(f'{split}: {len(files)} label files, {nonempty} non-empty')